# Phase 0 — Data Discovery: アリエールジェル Pricing Strategy

**Purpose:** Validate all data dimensions before main analysis.

| Check | Description |
|-------|-------------|
| 0-1 | Discover `jp_segment_4_name` for ｱﾘｴｰﾙｼﾞｪﾙ and ｱﾀｯｸ抗菌EX |
| 0-2 | Validate ASP = `pos_sales_amt / pos_unit_sales_qty` — nulls, zeros, outliers |
| 0-3 | Find renewal breakpoint (Apr/May 2025) via ASP and unit trends |
| 0-4 | `jp_prod_family_1_name` values for ｱﾘｴｰﾙｼﾞｪﾙ |
| 0-5 | Package weight/capacity from prod_dim for P&G sizes |

**Created:** 2026-02-19

---
## 0. Imports & Connection Setup

In [1]:
import os
import pandas as pd
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import numpy as np
from dotenv import load_dotenv
import databricks.sql as sql

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# ── Japanese font setup ────────────────────────────────────────────────
def _find_japanese_font() -> str | None:
    candidates = [
        'MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo',
        'IPAexGothic', 'IPAGothic', 'Noto Sans CJK JP',
        'Hiragino Sans', 'Hiragino Kaku Gothic Pro',
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font set: {_jp_font}')
else:
    print('⚠️  No Japanese font found')

# ── Databricks credentials ──────────────────────────────────────────────
load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')

assert DATABRICKS_HOST,      'DATABRICKS_HOST not set in .env'
assert DATABRICKS_TOKEN,     'DATABRICKS_TOKEN not set in .env'
assert DATABRICKS_HTTP_PATH, 'DATABRICKS_HTTP_PATH not set in .env'
print('✅ Credentials loaded')

# ── Reusable query helper ──────────────────────────────────────────────
def execute_query(query: str) -> pd.DataFrame:
    """Execute SQL against Databricks and return a DataFrame."""
    with sql.connect(
        server_hostname=DATABRICKS_HOST,
        http_path=DATABRICKS_HTTP_PATH,
        access_token=DATABRICKS_TOKEN
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            result  = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font set: MS Gothic
✅ Credentials loaded


---
## 1. Analysis Parameters

In [2]:
# ── Target brands ─────────────────────────────────────────────────────
ARIEL_GEL_SUB_BRAND   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX_SUB_BRAND   = 'ｱﾀｯｸ抗菌EX'
SUB_CATEGORY_FILTER   = '洗濯洗剤'
CATEGORY_FILTER       = 'Laundry'

# ── Analysis time window ──────────────────────────────────────────────
ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'

# ── Retailer codes (all national) ─────────────────────────────────────
RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN_CLAUSE = ', '.join(f"'{c}'" for c in RETAILER_CODES)

print('📋 Analysis Parameters')
print(f'  Ariel sub-brand  : {ARIEL_GEL_SUB_BRAND}')
print(f'  Attack sub-brand : {ATTACK_EX_SUB_BRAND}')
print(f'  Time window      : {ANALYSIS_START} → {ANALYSIS_END}')
print(f'  Retailers        : {len(RETAILER_CODES)} national retailers')

📋 Analysis Parameters
  Ariel sub-brand  : ｱﾘｴｰﾙｼﾞｪﾙ
  Attack sub-brand : ｱﾀｯｸ抗菌EX
  Time window      : 2025-01-01 → 2026-01-31
  Retailers        : 9 national retailers


---
### 📏 Canonical Definitions (Cross-Notebook Standard)

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). This is a rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes at least one subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. Label: `ASP (50 JPY bin)`. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

> ⚠️ These definitions are enforced consistently across notebooks 00–05 in this analysis.

---
## 2. Check 0-1: Discover Size Codes (`jp_segment_4_name`)

In [3]:
# ── Discover all sizes for both brands ────────────────────────────────
size_discovery_query = f"""
SELECT
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_segment_4_name           AS size_code,
    COUNT(DISTINCT idpos.shopper_key) AS shoppers,
    COUNT(DISTINCT idpos.transact_id) AS transactions,
    SUM(idpos.pos_unit_sales_qty)     AS total_units,
    SUM(idpos.pos_sales_amt)          AS total_sales
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN_CLAUSE})
  AND prod.jp_category_name = '{CATEGORY_FILTER}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CATEGORY_FILTER}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL_SUB_BRAND}', '{ATTACK_EX_SUB_BRAND}')
  AND shopper.member_ind = 'Y'
GROUP BY 1, 2
ORDER BY 1, 6 DESC
"""

print('⏳ Discovering size codes...', flush=True)
df_sizes = execute_query(size_discovery_query)
for col in ['shoppers', 'transactions', 'total_units', 'total_sales']:
    df_sizes[col] = pd.to_numeric(df_sizes[col])

print(f'\n✅ Found {len(df_sizes)} brand × size combinations\n')

# ── Display Ariel Gel sizes ────────────────────────────────────────────
print('=' * 70)
print('アリエールジェル — Size Codes')
print('=' * 70)
ariel_sizes = df_sizes[df_sizes['sub_brand'] == ARIEL_GEL_SUB_BRAND].copy()
ariel_sizes['sales_share_%'] = (ariel_sizes['total_sales'] / ariel_sizes['total_sales'].sum() * 100).round(1)
print(ariel_sizes.to_string(index=False))

print()
print('=' * 70)
print('アタック抗菌EX — Size Codes')
print('=' * 70)
attack_sizes = df_sizes[df_sizes['sub_brand'] == ATTACK_EX_SUB_BRAND].copy()
attack_sizes['sales_share_%'] = (attack_sizes['total_sales'] / attack_sizes['total_sales'].sum() * 100).round(1)
print(attack_sizes.to_string(index=False))

⏳ Discovering size codes...


HTTP request failed after retries: HTTPSConnectionPool(host='https', port=443): Max retries exceeded with url: //adb-2258763851730787.7.azuredatabricks.net/api/2.0/connector-service/feature-flags/PYTHON/4.2.3 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F8D62F76A0>: Failed to resolve 'https' ([Errno 11001] getaddrinfo failed)"))



✅ Found 15 brand × size combinations

アリエールジェル — Size Codes
sub_brand     size_code  shoppers  transactions  total_units      total_sales  sales_share_%
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   1602085        480628 4,687,628.00 4,022,636,319.00          39.20
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ    499695          9999 2,154,986.00 2,074,392,796.00          20.20
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大   1691773         81993 5,037,224.00 1,681,515,551.00          16.40
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   1073603         83901 2,356,268.00 1,634,782,986.00          15.90
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常   1008911        103430 2,413,186.00   605,426,106.00           5.90
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ     74715         39293   115,930.00   232,687,529.00           2.30
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ     15773          9094    22,293.00    15,766,221.00           0.20
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常       248            92       782.00       162,462.00           0.00

アタック抗菌EX — Size Codes
sub_brand     size_code  shoppers  transactions   total_units      total_sales 

---
## 3. Check 0-2: Validate ASP Data Quality

In [4]:
# ── ASP validation per brand × size ───────────────────────────────────
asp_validation_query = f"""
SELECT
    prod.jp_sub_brand_alter_lang_name                   AS sub_brand,
    prod.jp_segment_4_name                              AS size_code,
    COUNT(*)                                            AS total_rows,
    SUM(CASE WHEN idpos.pos_sales_amt IS NULL THEN 1 ELSE 0 END)      AS null_sales,
    SUM(CASE WHEN idpos.pos_unit_sales_qty IS NULL THEN 1 ELSE 0 END) AS null_units,
    SUM(CASE WHEN idpos.pos_unit_sales_qty = 0 THEN 1 ELSE 0 END)    AS zero_units,
    AVG(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS avg_asp,
    PERCENTILE_APPROX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0), 0.5) AS median_asp,
    MIN(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS min_asp,
    MAX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS max_asp,
    STDDEV(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0)) AS stddev_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN_CLAUSE})
  AND prod.jp_category_name = '{CATEGORY_FILTER}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CATEGORY_FILTER}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL_SUB_BRAND}', '{ATTACK_EX_SUB_BRAND}')
  AND shopper.member_ind = 'Y'
GROUP BY 1, 2
ORDER BY 1, 7 DESC
"""

print('⏳ Validating ASP data quality...', flush=True)
df_asp_dq = execute_query(asp_validation_query)
for col in df_asp_dq.columns[2:]:
    df_asp_dq[col] = pd.to_numeric(df_asp_dq[col])

print(f'\n✅ ASP validation complete\n')

# ── Display results ────────────────────────────────────────────────────
print('ASP Data Quality Summary (pos_sales_amt / pos_unit_sales_qty):')
print('=' * 100)
display_cols = ['sub_brand', 'size_code', 'total_rows', 'null_sales', 'null_units',
                'zero_units', 'avg_asp', 'median_asp', 'min_asp', 'max_asp']
print(df_asp_dq[display_cols].to_string(index=False))

# ── Flag issues ────────────────────────────────────────────────────────
issues = df_asp_dq[(df_asp_dq['null_sales'] > 0) | (df_asp_dq['null_units'] > 0) | (df_asp_dq['zero_units'] > 0)]
if len(issues) > 0:
    print('\n⚠️  Data quality issues found:')
    print(issues[['sub_brand', 'size_code', 'null_sales', 'null_units', 'zero_units']].to_string(index=False))
else:
    print('\n✅ No null/zero issues in ASP calculation')

# ── Flag outliers (ASP < 50 or > 5000) ────────────────────────────────
outliers = df_asp_dq[(df_asp_dq['min_asp'] < 50) | (df_asp_dq['max_asp'] > 5000)]
if len(outliers) > 0:
    print('\n⚠️  Potential ASP outliers (min<50 or max>5000):')
    print(outliers[['sub_brand', 'size_code', 'min_asp', 'max_asp']].to_string(index=False))
else:
    print('✅ No extreme ASP outliers detected')

⏳ Validating ASP data quality...


HTTP request error: 'NoneType' object has no attribute 'request'



✅ ASP validation complete

ASP Data Quality Summary (pos_sales_amt / pos_unit_sales_qty):
sub_brand     size_code  total_rows  null_sales  null_units  zero_units  avg_asp  median_asp  min_asp  max_asp
 ｱﾀｯｸ抗菌EX           ｿﾉﾀ       30393           0           0         148 2,749.26    3,200.00     0.00 4,291.00
 ｱﾀｯｸ抗菌EX   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     2052559           0           0        5401   893.14      891.00    73.00 1,073.00
 ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     9077734           0           0        6642   779.13      766.00    -1.00 1,659.00
 ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     4551366           0           0        3471   595.17      593.00    -1.00 1,329.00
 ｱﾀｯｸ抗菌EX         詰替超特大     5368910           0           0        4055   354.08      370.00  -359.00   759.00
 ｱﾀｯｸ抗菌EX          本体通常     1355584           0           0        1511   289.66      251.00     0.00   698.00
 ｱﾀｯｸ抗菌EX          詰替通常          11           0           0           0   130.95      138.00    82.00   145.57
ｱﾘｴｰﾙｼﾞｪﾙ           ｿ

HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'


---
## 4. Check 0-3: Find Renewal Breakpoint (ASP & Unit Trends by Month)

In [5]:
# ── Monthly ASP and units trend per brand × size ──────────────────────
monthly_trend_query = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS month,
    prod.jp_sub_brand_alter_lang_name                   AS sub_brand,
    prod.jp_segment_4_name                              AS size_code,
    COUNT(DISTINCT idpos.shopper_key)                   AS shoppers,
    SUM(idpos.pos_unit_sales_qty)                       AS total_units,
    SUM(idpos.pos_sales_amt)                            AS total_sales,
    AVG(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0)) AS avg_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN_CLAUSE})
  AND prod.jp_category_name = '{CATEGORY_FILTER}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CATEGORY_FILTER}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL_SUB_BRAND}', '{ATTACK_EX_SUB_BRAND}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Fetching monthly trends...', flush=True)
df_monthly = execute_query(monthly_trend_query)
df_monthly['month'] = pd.to_datetime(df_monthly['month'])
for col in ['shoppers', 'total_units', 'total_sales', 'avg_asp']:
    df_monthly[col] = pd.to_numeric(df_monthly[col])

print(f'✅ {len(df_monthly)} rows fetched')

⏳ Fetching monthly trends...
✅ 182 rows fetched


In [6]:
# ── Visualize ASP trend to find renewal breakpoint ─────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ariel Gel ASP trend by size
ariel_monthly = df_monthly[df_monthly['sub_brand'] == ARIEL_GEL_SUB_BRAND].copy()
ariel_monthly['month_str'] = ariel_monthly['month'].dt.strftime('%Y-%m')

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['アリエールジェル Monthly ASP by Size — Spot the Renewal Breakpoint',
                                    'アリエールジェル Monthly Shoppers by Size'],
                    vertical_spacing=0.12)

colors = px.colors.qualitative.Set2
sizes_list = ariel_monthly['size_code'].unique()

for i, size in enumerate(sizes_list):
    subset = ariel_monthly[ariel_monthly['size_code'] == size]
    color = colors[i % len(colors)]
    fig.add_trace(
        go.Scatter(x=subset['month'], y=subset['avg_asp'], name=size,
                   line=dict(color=color), legendgroup=size),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=subset['month'], y=subset['shoppers'], name=size,
                   line=dict(color=color, dash='dot'), legendgroup=size,
                   showlegend=False),
        row=2, col=1
    )

# Add vertical lines for Apr/May 2025 potential breakpoint
import datetime
for month_dt in [datetime.datetime(2025, 4, 1), datetime.datetime(2025, 5, 1)]:
    fig.add_vline(x=month_dt, line_dash='dash', line_color='red', opacity=0.5, row=1, col=1)
    fig.add_vline(x=month_dt, line_dash='dash', line_color='red', opacity=0.5, row=2, col=1)

fig.update_layout(height=700, title_text='Phase 0-3: Renewal Breakpoint Detection',
                  template='plotly_white')
fig.update_yaxes(title_text='ASP (JPY)', row=1, col=1)
fig.update_yaxes(title_text='Shoppers', row=2, col=1)
fig.show()

In [7]:
# ── Same chart for Attack 抗菌EX ──────────────────────────────────────
attack_monthly = df_monthly[df_monthly['sub_brand'] == ATTACK_EX_SUB_BRAND].copy()

if len(attack_monthly) > 0:
    fig2 = make_subplots(rows=2, cols=1,
                         subplot_titles=['アタック抗菌EX Monthly ASP by Size',
                                         'アタック抗菌EX Monthly Shoppers by Size'],
                         vertical_spacing=0.12)

    attack_sizes_list = attack_monthly['size_code'].unique()
    for i, size in enumerate(attack_sizes_list):
        subset = attack_monthly[attack_monthly['size_code'] == size]
        color = colors[i % len(colors)]
        fig2.add_trace(
            go.Scatter(x=subset['month'], y=subset['avg_asp'], name=size,
                       line=dict(color=color), legendgroup=size),
            row=1, col=1
        )
        fig2.add_trace(
            go.Scatter(x=subset['month'], y=subset['shoppers'], name=size,
                       line=dict(color=color, dash='dot'), legendgroup=size,
                       showlegend=False),
            row=2, col=1
        )

    fig2.update_layout(height=700, title_text='アタック抗菌EX: ASP & Shoppers Trend',
                       template='plotly_white')
    fig2.update_yaxes(title_text='ASP (JPY)', row=1, col=1)
    fig2.update_yaxes(title_text='Shoppers', row=2, col=1)
    fig2.show()
else:
    print('⚠️ No アタック抗菌EX data found — check sub_brand filter')

---
## 5. Check 0-4: Product Family Names (`jp_prod_family_1_name`)

In [8]:
# ── Discover jp_prod_family_1_name for both brands ────────────────────
prod_family_query = f"""
SELECT DISTINCT
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_prod_family_1_name        AS prod_family,
    prod.jp_segment_4_name            AS size_code,
    prod.jp_prod_form_name            AS prod_form
FROM id_pos_ai_1.prod_dim_ext_vw prod
WHERE prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL_SUB_BRAND}', '{ATTACK_EX_SUB_BRAND}')
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CATEGORY_FILTER}'
ORDER BY 1, 2, 3
"""

print('⏳ Fetching product family names...', flush=True)
df_prod_family = execute_query(prod_family_query)
print(f'\n✅ {len(df_prod_family)} distinct product family × size combinations\n')

print('アリエールジェル Product Families:')
print('=' * 70)
ariel_fam = df_prod_family[df_prod_family['sub_brand'] == ARIEL_GEL_SUB_BRAND]
print(ariel_fam.to_string(index=False))

print()
print('アタック抗菌EX Product Families:')
print('=' * 70)
attack_fam = df_prod_family[df_prod_family['sub_brand'] == ATTACK_EX_SUB_BRAND]
print(attack_fam.to_string(index=False))

⏳ Fetching product family names...

✅ 108 distinct product family × size combinations

アリエールジェル Product Families:
sub_brand               prod_family     size_code prod_form
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel          本体通常     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel          詰替特大     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel         詰替超特大     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel     詰替超ｼﾞｬﾝﾎﾞ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel          詰替通常     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel           ｿﾉﾀ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ            Ariel Gel_Biko          本体通常     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ            Ariel Gel_Biko         詰替超特大     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ            Ariel Gel_Biko     詰替超ｼﾞｬﾝﾎﾞ     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ            Ariel Gel_Biko          詰替通常     液体非濃縮
ｱﾘｴｰﾙｼﾞｪﾙ            Ariel Gel_Biko  詰替ｳﾙﾄﾗｼﾞｬ

---
## 6. Check 0-5: Package Weight/Capacity from Product Dimension

In [9]:
# ── Package capacity for P&G (Ariel Gel) from prod_dim ────────────────
# Query capacity/size information
capacity_query = f"""
SELECT
    jp_sub_brand_alter_lang_name,
    jp_prod_family_1_name,
    jp_segment_4_name,
    jp_size_name,
    jp_pack_size_name,
    jp_size_group_01_name,
    jp_size_group_02_name,
    jp_prod_alter_lang_name,
    jp_item_gtin
FROM id_pos_ai_1.prod_dim_ext_vw
WHERE jp_sub_brand_alter_lang_name = '{ARIEL_GEL_SUB_BRAND}'
  AND jp_sub_category_alter_lang_name = '{SUB_CATEGORY_FILTER}'
LIMIT 100
"""

print('⏳ Fetching product details for capacity info...', flush=True)
df_capacity = execute_query(capacity_query)
print(f'\n✅ {len(df_capacity)} products found')
print()
# Show raw data to understand what's available
print(df_capacity.head(10).to_string(index=False))
print()
# Check nulls
print('\\nNull counts per column:')
print(df_capacity.isnull().sum())
print('\\nUnique values per column:')
for col in df_capacity.columns:
    vals = df_capacity[col].dropna().unique()[:5]
    print(f'  {col}: {len(df_capacity[col].dropna().unique())} unique → {list(vals)}')

⏳ Fetching product details for capacity info...

✅ 100 products found

jp_sub_brand_alter_lang_name     jp_prod_family_1_name jp_segment_4_name   jp_size_name jp_pack_size_name jp_size_group_01_name jp_size_group_02_name                                   jp_prod_alter_lang_name   jp_item_gtin
                   ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel               ｿﾉﾀ             詰替              None                  None                  None                    ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ SP 詰替 超特大 増量品 1.45kg 04902430761376
                   ｱﾘｴｰﾙｼﾞｪﾙ                 Ariel Gel              詰替通常             詰替              None                  None                  None                    ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｻｲｴﾝｽﾌﾟﾗｽ 詰替 増量 790g 04902430816274
                   ｱﾘｴｰﾙｼﾞｪﾙ      Ariel Gel_Indoor Dry               ｿﾉﾀ ﾊﾞﾝﾄﾞﾙ(異ﾅﾙSKU)              None                  None                  None ｱﾘｴｰﾙ ﾘﾋﾞﾝｸﾞﾄﾞﾗｲ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ＋洗ﾀｸ槽ｸﾘｰﾅｰ ﾍﾟｱﾊﾟｯｸ 910ml+250g 04902430839501
                   ｱﾘｴｰﾙｼﾞｪﾙ      Ari

---
## 7. Summary: Confirmed Parameters for Main Analysis

Review the outputs above and confirm:
1. **Size codes** to use for Trial sizes vs. Repeat sizes
2. **ASP** calculation is clean and usable
3. **Renewal breakpoint** month (Apr or May 2025)
4. **Product family** values for variant-level analysis
5. **Capacity data** availability for per-dose pricing

In [10]:
# ── Store discovery results as variables for downstream notebooks ──────
print('=' * 70)
print('DISCOVERY SUMMARY')
print('=' * 70)

print('\n📦 Ariel Gel Size Codes Found:')
for _, row in ariel_sizes.iterrows():
    print(f"  {row['size_code']:30s} | {row['shoppers']:>8,.0f} shoppers | ¥{row['total_sales']:>12,.0f} sales")

print('\n📦 Attack 抗菌EX Size Codes Found:')
for _, row in attack_sizes.iterrows():
    print(f"  {row['size_code']:30s} | {row['shoppers']:>8,.0f} shoppers | ¥{row['total_sales']:>12,.0f} sales")

print('\n📊 ASP Quality:')
total_null = df_asp_dq['null_sales'].sum() + df_asp_dq['null_units'].sum()
total_zero = df_asp_dq['zero_units'].sum()
print(f"  Total null values: {total_null}")
print(f"  Total zero units:  {total_zero}")

print('\n🔄 Renewal Breakpoint:')
print('  Inspect the ASP trend charts above to confirm the exact month.')
print('  Expected: Apr/May 2025 based on business input.')

print('\n✅ Phase 0 data discovery complete — proceed to Phase 1')

DISCOVERY SUMMARY

📦 Ariel Gel Size Codes Found:
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                  | 1,602,085 shoppers | ¥4,022,636,319 sales
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                    |  499,695 shoppers | ¥2,074,392,796 sales
  詰替超特大                          | 1,691,773 shoppers | ¥1,681,515,551 sales
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   | 1,073,603 shoppers | ¥1,634,782,986 sales
  本体通常                           | 1,008,911 shoppers | ¥ 605,426,106 sales
  ｿﾉﾀ                            |   74,715 shoppers | ¥ 232,687,529 sales
  詰替超ｼﾞｬﾝﾎﾞ                      |   15,773 shoppers | ¥  15,766,221 sales
  詰替通常                           |      248 shoppers | ¥     162,462 sales

📦 Attack 抗菌EX Size Codes Found:
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                  | 3,445,967 shoppers | ¥8,238,781,951 sales
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   | 2,153,011 shoppers | ¥2,971,232,907 sales
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                    |  885,565 shoppers | ¥2,886,014,248 sales
  詰替超特大                          | 2,255,344 shoppers | ¥2,156,102,506 sales
  本